# Demo 3: Joins, CTEs, windows, and performance

This demo combines multiple tables and shows patterns used in real analytics work. Run demo 1 first to set up the database.

## Step 0: Connect to database

In [1]:
%load_ext sql
%config SqlMagic.autopandas = True
%sql duckdb:///demo_chinook.duckdb?access_mode=read_only

Connecting to 'duckdb:///demo_chinook.duckdb?access_mode=read_only'

## Step 1: Join tracks to artists

In [2]:
%%sql
SELECT t.Name AS track_name,
    a.Name AS artist_name,
    t.UnitPrice
FROM tracks AS t
INNER JOIN albums AS al
    ON t.AlbumId = al.AlbumId
INNER JOIN artists AS a
    ON al.ArtistId = a.ArtistId
ORDER BY t.UnitPrice DESC, t.Name
LIMIT 10;

Running query in 'duckdb:///demo_chinook.duckdb?access_mode=read_only'

,track_name,artist_name,UnitPrice
0,"""?""",Lost,1.99
1,...And Found,Lost,1.99
2,...In Translation,Lost,1.99
3,.07%,Heroes,1.99
4,"A Benihana Christmas, Pts. 1 & 2",The Office,1.99
5,A Day In the Life,Battlestar Galactica,1.99
6,A Measure of Salvation,Battlestar Galactica,1.99
7,A Tale of Two Cities,Lost,1.99
8,Abandoned,Lost,1.99
9,Adrift,Lost,1.99


Checkpoint: results should show track names alongside artist names.

Example output:

| track_name | artist_name | UnitPrice |
| --- | --- | --- |
| The Woman King | Iron Maiden | 1.99 |
| The Trooper | Iron Maiden | 1.99 |

## Step 2: CTE to define a cohort of high-value customers

In [3]:
%%sql
WITH customer_spend AS (
    SELECT CustomerId, SUM(Total) AS total_spend
    FROM invoices
    GROUP BY CustomerId
),
high_value AS (
    SELECT CustomerId
    FROM customer_spend
    WHERE total_spend >= 40
)
SELECT c.FirstName, c.LastName, c.Country, cs.total_spend
FROM customers AS c
INNER JOIN customer_spend AS cs
    ON c.CustomerId = cs.CustomerId
INNER JOIN high_value AS h
    ON c.CustomerId = h.CustomerId
ORDER BY cs.total_spend DESC;

Running query in 'duckdb:///demo_chinook.duckdb?access_mode=read_only'

,FirstName,LastName,Country,total_spend
0,Helena,Holý,Czech Republic,49.62
1,Richard,Cunningham,USA,47.62
2,Luis,Rojas,Chile,46.62
3,Ladislav,Kovács,Hungary,45.62
4,Hugh,O'Reilly,Ireland,45.62
5,Julia,Barnett,USA,43.62
6,Frank,Ralston,USA,43.62
7,Fynn,Zimmermann,Germany,43.62
8,Astrid,Gruber,Austria,42.62
9,Victor,Stevens,USA,42.62


Checkpoint: results should show only customers with total spend >= 40.

Example output:

| FirstName | LastName | Country | total_spend |
| --- | --- | --- | --- |
| Victor | Stevens | USA | 49.62 |
| Astrid | Gruber | Germany | 42.62 |

## Step 2b: IN vs EXISTS

In [4]:
%%sql
SELECT c.CustomerId, c.FirstName, c.LastName
FROM customers AS c
WHERE c.CustomerId IN (
    SELECT CustomerId
    FROM invoices
);

Running query in 'duckdb:///demo_chinook.duckdb?access_mode=read_only'

,CustomerId,FirstName,LastName
0,11,Alexandre,Rocha
1,16,Frank,Harris
2,26,Richard,Cunningham
3,55,Mark,Taylor
4,15,Jennifer,Peterson
5,22,Heather,Leacock
6,31,Martha,Silk
7,32,Aaron,Mitchell
8,52,Emma,Jones
9,35,Madalena,Sampaio


In [5]:
%%sql
SELECT c.CustomerId, c.FirstName, c.LastName
FROM customers AS c
WHERE EXISTS (
    SELECT 1
    FROM invoices AS i
    WHERE i.CustomerId = c.CustomerId
);

Running query in 'duckdb:///demo_chinook.duckdb?access_mode=read_only'

,CustomerId,FirstName,LastName
0,1,Luís,Gonçalves
1,2,Leonie,Köhler
2,3,François,Tremblay
3,4,Bjørn,Hansen
4,5,František,Wichterlová
5,6,Helena,Holý
6,7,Astrid,Gruber
7,8,Daan,Peeters
8,9,Kara,Nielsen
9,10,Eduardo,Martins


Checkpoint: both queries return the same customers.

## Step 3: Window function for running totals

In [6]:
%%sql
SELECT InvoiceId,
    InvoiceDate,
    Total,
    SUM(Total) OVER (
        ORDER BY InvoiceDate
    ) AS running_total
FROM invoices
ORDER BY InvoiceDate
LIMIT 20;

Running query in 'duckdb:///demo_chinook.duckdb?access_mode=read_only'

,InvoiceId,InvoiceDate,Total,running_total
0,1,2009-01-01,1.98,1.98
1,2,2009-01-02,3.96,5.94
2,3,2009-01-03,5.94,11.88
3,4,2009-01-06,8.91,20.79
4,5,2009-01-11,13.86,34.65
5,6,2009-01-19,0.99,35.64
6,7,2009-02-01,1.98,39.60
7,8,2009-02-01,1.98,39.60
8,9,2009-02-02,3.96,43.56
9,10,2009-02-03,5.94,49.50


Checkpoint: `running_total` should increase over time.

## Step 3b: Plot running totals (Altair)

In [7]:
%%sql running_totals <<
SELECT InvoiceDate, Total,
    SUM(Total) OVER (ORDER BY InvoiceDate) AS running_total
FROM invoices
ORDER BY InvoiceDate

Running query in 'duckdb:///demo_chinook.duckdb?access_mode=read_only'

In [8]:
import altair as alt

chart = (
    alt.Chart(running_totals)
    .mark_line()
    .encode(
        x=alt.X("InvoiceDate:T", title="Invoice date"),
        y=alt.Y("running_total:Q", title="Running total"),
        tooltip=["InvoiceDate", "running_total"]
    )
)

chart

alt.Chart(...)

Checkpoint: the line should increase over time.

## Step 4: Create a view for reuse

In [9]:
%%sql
DROP VIEW IF EXISTS top_tracks;

CREATE VIEW top_tracks AS
SELECT t.TrackId, t.Name, t.UnitPrice
FROM tracks AS t
ORDER BY t.UnitPrice DESC;

Running query in 'duckdb:///demo_chinook.duckdb?access_mode=read_only'

RuntimeError: (_duckdb.InvalidInputException) Invalid Input Error: Cannot execute statement of type "DROP" on database "demo_chinook" which is attached in read-only mode!
[SQL: DROP VIEW IF EXISTS top_tracks;]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [ ]:
%%sql
SELECT * FROM top_tracks LIMIT 5;

Running query in 'duckdb:///demo_chinook.duckdb'

TrackId,Name,UnitPrice
2819,Battlestar Galactica: The Story So Far,1.99
2820,Occupation / Precipice,1.99
2821,"Exodus, Pt. 1",1.99
2822,"Exodus, Pt. 2",1.99
2823,Collaborators,1.99


Checkpoint: results should show the most expensive tracks.

## Step 5: Inspect a query plan

In [ ]:
%%sql
EXPLAIN
SELECT BillingCountry, COUNT(*) AS invoice_count
FROM invoices
GROUP BY BillingCountry;

Running query in 'duckdb:///demo_chinook.duckdb'

explain_key,explain_value


Checkpoint: output should be a query plan (not a regular result table).

## Step 6: DATE_TRUNC for time buckets

In [ ]:
%%sql
SELECT DATE_TRUNC('month', InvoiceDate) AS month, COUNT(*) AS invoice_count
FROM invoices
GROUP BY month
ORDER BY month;

Running query in 'duckdb:///demo_chinook.duckdb'

month,invoice_count
2009-01-01,6
2009-02-01,7
2009-03-01,7
2009-04-01,7
2009-05-01,7
2009-06-01,7
2009-07-01,7
2009-08-01,7
2009-09-01,7
2009-10-01,7


Checkpoint: one row per month bucket.